In [48]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import json
from pathlib import Path
from sklearn.model_selection import train_test_split,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.metrics import mean_absolute_error,r2_score
from xgboost import XGBRegressor
from datetime import datetime,timezone

In [2]:
root_dir = Path.cwd().parent
data_dir = root_dir / 'data' / 'interim' / 'urbaneats-cleaned-dataset.csv'

In [3]:
df = pd.read_csv(data_dir)

In [4]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,pickup_time_minutes,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,Saturday,1,11.0,15.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,Friday,0,19.0,5.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,Saturday,1,8.0,15.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,Tuesday,0,18.0,10.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,Saturday,1,13.0,15.0,afternoon,6.210138,medium


In [5]:
df.shape

(45502, 27)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

In [8]:
# check for missing values

df.isna().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [9]:
dagshub.init(repo_owner='AvanindraBose', repo_name='Urban-Eats-Food-Delivery-Time-Prediction', mlflow=True)

Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction"

Repository AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction initialized!

In [10]:
mlflow.set_tracking_uri('https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow')

# Droping Missing Values and then Selecting the Best HyperParameters for XGB Regressor.

In [11]:
temp_df = df.copy().dropna()

In [12]:
temp_df.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [13]:
temp_df.shape

(37695, 16)

In [14]:
X = temp_df.drop(columns= ['time_taken'])
y = temp_df['time_taken']

In [15]:
X.sample(10)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
11746,27.0,4.6,sunny,low,1,meal,motorcycle,1.0,no,metropolitian,0,10.0,night,20.179004,very_long
13201,26.0,4.8,stormy,low,1,snack,scooter,0.0,no,metropolitian,0,10.0,morning,3.104556,short
40971,27.0,5.0,cloudy,medium,1,snack,scooter,1.0,no,metropolitian,0,10.0,afternoon,13.680927,long
23898,38.0,4.3,stormy,jam,2,drinks,motorcycle,2.0,yes,metropolitian,0,10.0,evening,17.078934,very_long
9921,25.0,5.0,sunny,low,2,meal,electric_scooter,0.0,no,urban,0,15.0,night,12.420599,long
27448,27.0,4.7,windy,medium,1,buffet,motorcycle,1.0,no,urban,0,10.0,evening,4.476869,short
9797,39.0,4.8,stormy,low,1,drinks,motorcycle,1.0,no,metropolitian,0,15.0,morning,2.937443,short
16926,26.0,4.6,sandstorms,jam,2,snack,scooter,1.0,no,metropolitian,1,5.0,evening,7.790337,medium
21283,34.0,4.7,sunny,medium,0,drinks,motorcycle,1.0,no,urban,1,15.0,afternoon,4.657655,short
19470,22.0,4.6,sunny,low,0,buffet,motorcycle,1.0,no,metropolitian,1,5.0,night,19.975696,very_long


In [16]:
y.sample(10)

42604    40
22842    29
42602    29
6915     12
31629    45
5688     22
19308    30
36842    21
23587    43
36371    31
Name: time_taken, dtype: int64

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [19]:
num_cols = X_train.select_dtypes(include=np.number).columns.to_list()

In [20]:
num_cols.remove('vehicle_condition')
num_cols.remove('multiple_deliveries')

In [21]:
X_train.select_dtypes(include=object).columns.to_list()

['weather',
 'traffic',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'order_time_of_day',
 'distance_type']

In [22]:
ordinal_cat_cols = ['traffic','distance_type']

nominal_cat_cols = [
    'weather',
    'type_of_order',
    'type_of_vehicle',
    'festival',
    'city_type',
    'order_time_of_day'
]

In [23]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

13

In [24]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

__Testing the Pipeline.__

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
)

In [27]:
pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(random_state=42, n_jobs=-1))
    ])

model_pipe_tt = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

In [28]:
scores = cross_validate(
            model_pipe_tt,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [29]:
scores

{'fit_time': array([0.56143928, 0.53990722, 0.54390574, 0.5369072 , 0.54941487]),
 'score_time': array([0.0489037 , 0.05159807, 0.04911041, 0.05259943, 0.04760385]),
 'test_mae': array([-3.20097303, -3.1386447 , -3.13769221, -3.12094164, -3.12675405]),
 'train_mae': array([-2.56598568, -2.55687475, -2.5639298 , -2.54815674, -2.54575753]),
 'test_r2': array([0.81691056, 0.8251425 , 0.82638359, 0.82767761, 0.82721698]),
 'train_r2': array([0.88134819, 0.88192463, 0.88136882, 0.88318264, 0.88311261])}

__Conducting Hyoer Parameter Tuning.__

In [50]:
def build_model(params):

    preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**params))
    ])

    model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

    return model_pipe

In [51]:
def objective(trial):

    with mlflow.start_run(run_name=f"trial_{trial.number}",nested=True) as run:

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
            "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),   # L1
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True), # L2
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
            "tree_method": "hist",  # fast histogram-based tree building
            "random_state": 42,
            "n_jobs": -1
}

        model = build_model(params)

        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                'mae': 'neg_mean_absolute_error',
                'r2': 'r2'
            },
            n_jobs = -1,
            return_train_score = True
        )

        train_mae = -scores["train_mae"].mean()
        val_mae = -scores["test_mae"].mean()
        val_mae_std = scores["test_mae"].std()
        train_r2 = scores["train_r2"].mean()
        val_r2 = scores["test_r2"].mean()

        mlflow.log_param("model_type","XGBRegressor")
        mlflow.log_param("trial_number", trial.number)
        mlflow.log_params(trial.params)

        mlflow.log_metric("train_mae_mean", train_mae)
        mlflow.log_metric("val_mae_mean", val_mae)
        mlflow.log_metric("val_mae_std", val_mae_std)
        mlflow.log_metric("train_r2_mean", train_r2)
        mlflow.log_metric("val_r2_mean", val_r2)

        for i, score in enumerate(scores["test_mae"]):
            mlflow.log_metric(f"fold_{i}_val_mae", -score)

        for i, score in enumerate(scores["test_r2"]):
            mlflow.log_metric(f"fold_{i}_val_r2", score)

        trial.set_user_attr("val_mae", val_mae)
        trial.set_user_attr("val_r2", val_r2)

        return val_mae

In [56]:
mlflow.set_experiment("Exp 4 - XGB HP Tuning")

study = optuna.create_study(direction="minimize", study_name="XGB HP Tuning")

artifact_dir = Path.cwd()
trials_path = artifact_dir / "xgb_hp_optuna_trials.csv"
params_path = artifact_dir / "xgb_best_params.json"
config_path = artifact_dir / "xgb_preprocessing_config.json"

fixed_params = {
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

with mlflow.start_run(run_name="XGB HP Tuning") as parent_run:
    mlflow.set_tags({
        "n_trials": 35,
        "cv_folds": 5,
        "objective_metric": "val_mae",
        "model_type": "XGBRegressor",
        "created_by": "Avanindra Bose"
    })

    study.optimize(objective, n_trials=35)

    best_trial = study.best_trial
    best_params = {**best_trial.params, **fixed_params}

    best_model_pipe = build_model(best_params)
    best_model_pipe.fit(X_train, y_train)

    y_pred_train = best_model_pipe.predict(X_train)
    y_pred_test = best_model_pipe.predict(X_test)

    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.set_tag("best_trial_number", best_trial.number)
    mlflow.log_params({f"xgb__{k}": v for k, v in best_params.items()})

    mlflow.log_metrics({
        "best_cv_mae": best_trial.value,
        "final_train_mae": train_mae,
        "final_test_mae": test_mae,
        "final_train_r2": train_r2,
        "final_test_r2": test_r2,
    })

    trials_df = study.trials_dataframe()
    trials_df.to_csv(trials_path, index=False)

    preprocessing_config = {
        "missing_value_strategy": "dropna",
        "numerical_features": num_cols,
        "nominal_categorical_features": nominal_cat_cols,
        "ordinal_categorical_features": ordinal_cat_cols,
        "ordinal_categories": {
            "traffic": traffic_order,
            "distance_type": distance_type_order,
        },
        "numerical_scaler": "MinMaxScaler",
        "nominal_encoder": "OneHotEncoder(drop='first', handle_unknown='ignore')",
        "ordinal_encoder": "OrdinalEncoder",
        "target_transformer": "PowerTransformer",
    }

    with open(config_path, "w") as f:
        json.dump(preprocessing_config, f, indent=2)

    with open(params_path, "w") as f:
        json.dump(best_params, f, indent=2)

    mlflow.log_artifact(str(trials_path))
    mlflow.log_artifact(str(config_path))
    mlflow.log_artifact(str(params_path))

    mlflow.sklearn.log_model(
    sk_model=best_model_pipe,
    name="xgb_hp_tuned_model",
    input_example=X_train.iloc[:5],          
    signature=mlflow.models.infer_signature( 
        X_train, 
        best_model_pipe.predict(X_train)
        )
    )

[I 2026-06-04 19:01:18,617] A new study created in memory with name: XGB HP Tuning


🏃 View run trial_0 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/fb617c10a10946689aadc527a604883a
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5


[I 2026-06-04 19:01:35,460] Trial 0 finished with value: 6.62979097366333 and parameters: {'n_estimators': 225, 'max_depth': 8, 'learning_rate': 0.0010355013196794885, 'min_child_weight': 1, 'gamma': 4.176521826183241, 'subsample': 0.9300149384186771, 'colsample_bytree': 0.8513361437449145, 'colsample_bylevel': 0.8125469573987565, 'colsample_bynode': 0.5537404205883265, 'reg_alpha': 2.635090920994216e-08, 'reg_lambda': 2.7261571021144227e-08, 'max_delta_step': 1, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 6.62979097366333.


🏃 View run trial_1 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/7ee1e26bf5ed4b908c5f243a75461e28
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5


[I 2026-06-04 19:02:19,467] Trial 1 finished with value: 3.1665180683135987 and parameters: {'n_estimators': 296, 'max_depth': 9, 'learning_rate': 0.25947613889762006, 'min_child_weight': 2, 'gamma': 1.6085463408586331, 'subsample': 0.8126869496677798, 'colsample_bytree': 0.9592070087603219, 'colsample_bylevel': 0.604933280130034, 'colsample_bynode': 0.8413090057527759, 'reg_alpha': 0.07312908259636056, 'reg_lambda': 1.6603117740506608e-06, 'max_delta_step': 8, 'grow_policy': 'lossguide'}. Best is trial 1 with value: 3.1665180683135987.


...

🏃 View run trial_33 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/d6247c34145c47a9b1b693ce25d6c332
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5


[I 2026-06-04 19:37:18,728] Trial 33 finished with value: 3.0465901851654054 and parameters: {'n_estimators': 587, 'max_depth': 9, 'learning_rate': 0.039634567177513826, 'min_child_weight': 6, 'gamma': 0.6058356338406817, 'subsample': 0.7591331610065966, 'colsample_bytree': 0.9644039422985538, 'colsample_bylevel': 0.9477142587934383, 'colsample_bynode': 0.7966500966291707, 'reg_alpha': 1.679189010062973e-07, 'reg_lambda': 1.8860641630342156, 'max_delta_step': 1, 'grow_policy': 'lossguide'}. Best is trial 27 with value: 3.022345781326294.


🏃 View run trial_34 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/85615548d94e4b9fa707556f78a757f5
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5


[I 2026-06-04 19:38:07,407] Trial 34 finished with value: 3.072011041641235 and parameters: {'n_estimators': 486, 'max_depth': 10, 'learning_rate': 0.012908085776128111, 'min_child_weight': 2, 'gamma': 1.6599883563273727, 'subsample': 0.8453863773170331, 'colsample_bytree': 0.968172043596212, 'colsample_bylevel': 0.7740179941235721, 'colsample_bynode': 0.869586357578519, 'reg_alpha': 4.2296754138381234e-07, 'reg_lambda': 0.11647228671659704, 'max_delta_step': 0, 'grow_policy': 'lossguide'}. Best is trial 27 with value: 3.022345781326294.
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\si

🏃 View run XGB HP Tuning at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5/runs/b49702f40cbd4581a53f8b131c566df7
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/5


In [59]:
best_trial.params

{'n_estimators': 603,
 'max_depth': 11,
 'learning_rate': 0.03417913351080966,
 'min_child_weight': 4,
 'gamma': 0.452621135308464,
 'subsample': 0.863443370509037,
 'colsample_bytree': 0.9651511117267577,
 'colsample_bylevel': 0.8881626404741021,
 'colsample_bynode': 0.7870334195129309,
 'reg_alpha': 8.213030683468667e-08,
 'reg_lambda': 0.2436995143218315,
 'max_delta_step': 1,
 'grow_policy': 'lossguide'}

In [57]:
best_params

{'n_estimators': 603,
 'max_depth': 11,
 'learning_rate': 0.03417913351080966,
 'min_child_weight': 4,
 'gamma': 0.452621135308464,
 'subsample': 0.863443370509037,
 'colsample_bytree': 0.9651511117267577,
 'colsample_bylevel': 0.8881626404741021,
 'colsample_bynode': 0.7870334195129309,
 'reg_alpha': 8.213030683468667e-08,
 'reg_lambda': 0.2436995143218315,
 'max_delta_step': 1,
 'grow_policy': 'lossguide',
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1}

In [60]:
best_trial.value

3.022345781326294

In [61]:
optuna.visualization.plot_optimization_history(study)

In [62]:
optuna.visualization.plot_param_importances(study)

In [63]:
optuna.visualization.plot_slice(study)